In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName("Joins")
    .master("local[*]")   
    .getOrCreate()
)

spark

In [2]:
spark.stop()

In [4]:
from pyspark.sql import functions as F

data = [
    (1, "Alice", "HR", "Mumbai", 50000, "2024-01-10"),
    (2, "Bob", "IT", "Bangalore", 70000, "2024-01-12"),
    (3, "Charlie", "IT", "Mumbai", 80000, "2024-01-15"),
    (4, "David", "HR", "Delhi", 60000, "2024-01-18"),
    (5, "Eve", "Sales", "Mumbai", 75000, "2024-01-20"),
    (6, "Frank", "Sales", "Delhi", 72000, "2024-01-22"),
    (7, "Grace", "IT", "Bangalore", 90000, "2024-01-25"),
    (8, "Helen", "HR", "Mumbai", 65000, "2024-01-28")
]

columns = ["id", "name", "department", "city", "salary", "joining_date"]

df = spark.createDataFrame(data, columns)

In [5]:
df.show(3)

+---+-------+----------+---------+------+------------+
| id|   name|department|     city|salary|joining_date|
+---+-------+----------+---------+------+------------+
|  1|  Alice|        HR|   Mumbai| 50000|  2024-01-10|
|  2|    Bob|        IT|Bangalore| 70000|  2024-01-12|
|  3|Charlie|        IT|   Mumbai| 80000|  2024-01-15|
+---+-------+----------+---------+------+------------+
only showing top 3 rows



In [12]:
from pyspark.sql.functions import col,lit
# 1. Count total number of employees
# 2. Count employees per department

# Top 2 depmartments by total sal 

# 9. Get top 3 highest paid employees
# 10. Count employees per city

#df.count()
#df.groupBy("department").count().show()
df.groupBy("department").agg(
    F.sum("salary").alias("total_salary"),   
).orderBy("total_salary",ascending = False).limit(2).show()

+----------+------------+
|department|total_salary|
+----------+------------+
|        IT|      240000|
|        HR|      175000|
+----------+------------+



In [15]:
df.orderBy(F.col("salary").asc()).select(F.col("salary").alias("ltoh")).show()

+-----+
| ltoh|
+-----+
|50000|
|60000|
|65000|
|70000|
|72000|
|75000|
|80000|
|90000|
+-----+



In [ ]:
df.groupBy("department").F.avg("salary").show()

+----------+------------------+
|department|       avg(salary)|
+----------+------------------+
|        HR|58333.333333333336|
|        IT|           80000.0|
|     Sales|           73500.0|
+----------+------------------+



In [18]:
df.groupBy("department").agg(
    F.avg("salary").alias("average_sal")
).filter(F.col("average_sal") > 70000).show()

+----------+-----------+
|department|average_sal|
+----------+-----------+
|        IT|    80000.0|
|     Sales|    73500.0|
+----------+-----------+



In [19]:
df.groupBy("city").agg(
    F.avg("salary").alias("low_sal")
).orderBy(F.col("low_sal")).limit(1).show()

+-----+-------+
| city|low_sal|
+-----+-------+
|Delhi|66000.0|
+-----+-------+



In [24]:
dept_sal=df.groupBy("department").agg(
    F.sum("salary").alias("dept_sal")
)
total_sal=df.agg(
    F.sum("salary").alias("total_salary")
)
result_df=dept_sal.crossJoin(total_sal).withColumn("percentage",
((F.col("dept_sal")/F.col("total_salary"))*100))

result_df.orderBy(F.col("percentage").desc()).show()                                                   

+----------+--------+------------+------------------+
|department|dept_sal|total_salary|        percentage|
+----------+--------+------------+------------------+
|        IT|  240000|      562000|42.704626334519574|
|        HR|  175000|      562000| 31.13879003558719|
|     Sales|  147000|      562000|26.156583629893237|
+----------+--------+------------+------------------+



In [25]:
spark.stop()